# Benchmarking of ML methods to analyze a dataset on the Estimation of obesity habits

### Short description of the dataset

This paper contains data for the estimation of obesity levels in people from the countries of Mexico,
Peru and Colombia, with ages between 14 and 61 and diverse eating habits and physical condition as
mentioned by [1].

Data was collected using a web platform with a survey (see Table 1) where anon-
ymous users answered each question, then the information was processed obtaining 17 attributes and
2111 records, after a balancing process described in Figs. 1 and 2.

### The attributes related with eating habits are: 

- Frequent consumption of high caloric food (FAVC),
- Frequency of consumption of vegetables (FCVC), 
- Number of main meals (NCP), 
- Consumption of food between meals (CAEC),
- Consumption of water daily (CH20),
- Consumption of alcohol (CALC). 

### The attributes related with the physical condition are: 

- Calories consumption monitoring (SCC), 
- Physical activity frequency (FAF), 
- Time using technology devices (TUE),
- Transportation used (MTRANS)

### other variables obtained were: 
- Gender, 
- Age,
- Height 
- Weight.

Finally, all data was labeled and the class variable NObesity was created with the
values of: 
- Insuf cient Weight, Normal Weight, Overweight Level I, Overweight Level II, Obesity Type I,
Obesity Type II and Obesity Type III, based on Equation (1) and information from WHO and Mexican
Normativity.

 The data contains numerical data and continous data, so it can be used for analysis based
on algorithms of classi cation, prediction, segmentation and association. Data is available in CSV
format and ARFF format to be used with the Weka tool.

# I - Data preprocessing and feature engineering

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import scipy
import seaborn as sns

In [7]:
df = pd.read_csv("data/ObesityDataSet_raw_and_data_sinthetic.csv")
df

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.000000,1.620000,64.000000,yes,no,2.0,3.0,Sometimes,no,2.000000,no,0.000000,1.000000,no,Public_Transportation,Normal_Weight
1,Female,21.000000,1.520000,56.000000,yes,no,3.0,3.0,Sometimes,yes,3.000000,yes,3.000000,0.000000,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.000000,1.800000,77.000000,yes,no,2.0,3.0,Sometimes,no,2.000000,no,2.000000,1.000000,Frequently,Public_Transportation,Normal_Weight
3,Male,27.000000,1.800000,87.000000,no,no,3.0,3.0,Sometimes,no,2.000000,no,2.000000,0.000000,Frequently,Walking,Overweight_Level_I
4,Male,22.000000,1.780000,89.800000,no,no,2.0,1.0,Sometimes,no,2.000000,no,0.000000,0.000000,Sometimes,Public_Transportation,Overweight_Level_II
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2106,Female,20.976842,1.710730,131.408528,yes,yes,3.0,3.0,Sometimes,no,1.728139,no,1.676269,0.906247,Sometimes,Public_Transportation,Obesity_Type_III
2107,Female,21.982942,1.748584,133.742943,yes,yes,3.0,3.0,Sometimes,no,2.005130,no,1.341390,0.599270,Sometimes,Public_Transportation,Obesity_Type_III
2108,Female,22.524036,1.752206,133.689352,yes,yes,3.0,3.0,Sometimes,no,2.054193,no,1.414209,0.646288,Sometimes,Public_Transportation,Obesity_Type_III
2109,Female,24.361936,1.739450,133.346641,yes,yes,3.0,3.0,Sometimes,no,2.852339,no,1.139107,0.586035,Sometimes,Public_Transportation,Obesity_Type_III


### Basic statistics about the data

In [8]:
df.head(5)

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [9]:
n, p =df.shape
n, p

(2111, 17)

In [10]:
## Rajout de la colonne IMC_index puis mise à jour des classes de poids

df["IMC_index"] = df["Weight"]/(df["Height"]**2)

def categorize_weight(row):
    weight_index = row['Weight'] / (row['Height'] ** 2)
    if weight_index < 18.5:
        return 'Underweight'
    elif 18.5 <= weight_index < 25:
        return 'Normal'
    elif 25 <= weight_index < 30:
        return 'Overweight'
    elif 30 <= weight_index < 35:
        return 'Obesity I'
    elif 35 <= weight_index < 40:
        return 'Obesity II'
    else:
        return 'Obesity III'
    
df["Weight_class"] = df.apply(categorize_weight, axis=1)
weight_order = ["Underweight", "Normal", "Overweight", "Obesity I", "Obesity II", "Obesity III"]
df["Weight_class"] = pd.Categorical(df["Weight_class"], categories=weight_order, ordered=True)

## Train / test separation

In [11]:
def split_train_test(data : pd.DataFrame, test_ratio, y):
    shuffle = np.random.permutation(n)
    separator = int(test_ratio*n)
    tr_indices = shuffle[:separator]
    te_indices = shuffle[separator:]

    return data.iloc[tr_indices], data.iloc[te_indices], y[tr_indices], y[te_indices]

In [12]:
tr_data, te_data, y_tr, y_te = split_train_test(df, 0.8, df["Weight_class"])
tr_data.head(10)

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad,IMC_index,Weight_class
1361,Female,18.000000,1.686904,90.004046,yes,yes,2.737149,3.000000,Sometimes,no,1.997195,no,1.352663,0.468483,Sometimes,Public_Transportation,Obesity_Type_I,31.628697,Obesity I
1517,Male,27.562425,1.908608,128.828122,yes,yes,2.206276,3.000000,Sometimes,no,1.813082,no,1.949840,0.000000,Sometimes,Public_Transportation,Obesity_Type_II,35.365288,Obesity II
1611,Male,24.634740,1.829757,120.980508,yes,yes,2.644094,3.000000,Sometimes,no,2.740525,no,2.000000,0.500936,Sometimes,Public_Transportation,Obesity_Type_II,36.135042,Obesity II
556,Male,18.000000,1.781543,50.869704,no,yes,2.052932,3.000000,Sometimes,no,2.000000,no,0.520408,1.000000,Sometimes,Public_Transportation,Insufficient_Weight,16.027530,Underweight
1020,Female,25.483381,1.565288,64.848627,yes,no,2.000000,1.000000,Sometimes,no,1.000000,no,0.740633,0.000000,no,Public_Transportation,Overweight_Level_II,26.467461,Overweight
1300,Female,22.591026,1.650012,87.676154,yes,yes,2.983851,1.068443,Sometimes,no,2.854161,no,1.103209,1.000000,no,Public_Transportation,Obesity_Type_I,32.203812,Obesity I
2066,Female,21.568951,1.699315,133.107610,yes,yes,3.000000,3.000000,Sometimes,no,1.733306,no,1.773304,0.921136,Sometimes,Public_Transportation,Obesity_Type_III,46.095136,Obesity III
534,Male,18.216032,1.755507,52.000000,yes,yes,3.000000,3.000000,Frequently,no,2.000000,no,0.658894,1.000000,no,Public_Transportation,Insufficient_Weight,16.873229,Underweight
1374,Male,25.955361,1.830384,105.479313,yes,yes,3.000000,3.000000,Sometimes,no,2.480277,no,1.796779,1.596562,Sometimes,Public_Transportation,Obesity_Type_I,31.483490,Obesity I
1549,Male,23.454558,1.754218,117.384745,yes,yes,1.617093,3.000000,Sometimes,no,2.000000,no,1.818052,0.631825,Sometimes,Public_Transportation,Obesity_Type_II,38.145607,Obesity II


## Pipeline : Categorigal & numerical variables transformation

In [16]:
numerical_variables = ["Age", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
categorical_variables = ["Gender", "family_history_with_overweight", "FAVC", "CAEC", "SMOKE", "SCC", "CALC", "MTRANS"]
labels = list(df["Weight_class"].unique())
print(df[categorical_variables])
print(labels)

      Gender family_history_with_overweight FAVC       CAEC SMOKE  SCC  \
0     Female                            yes   no  Sometimes    no   no   
1     Female                            yes   no  Sometimes   yes  yes   
2       Male                            yes   no  Sometimes    no   no   
3       Male                             no   no  Sometimes    no   no   
4       Male                             no   no  Sometimes    no   no   
...      ...                            ...  ...        ...   ...  ...   
2106  Female                            yes  yes  Sometimes    no   no   
2107  Female                            yes  yes  Sometimes    no   no   
2108  Female                            yes  yes  Sometimes    no   no   
2109  Female                            yes  yes  Sometimes    no   no   
2110  Female                            yes  yes  Sometimes    no   no   

            CALC                 MTRANS  
0             no  Public_Transportation  
1      Sometimes  Public_Tr

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [ ]:
num_pipeline = Pipeline([("imputer", SimpleImputer(strategy='median')),
                        ("std_scaler", StandardScaler())])
full_pipeline = ColumnTransformer([("num", num_pipeline, numerical_variables),
                                   ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_variables)], verbose_feature_names_out= False)
full_pipeline.set_output(transform="pandas")
tr_data = full_pipeline.fit_transform(tr_data)
te_data = full_pipeline.transform(te_data)

## II - ML Approaches comparisons

For each method we will work sequentally : 
- First, we will train a dummy classifier on the training data, evaluating it on the test set
- Then we will optimize the hyperparameters through N = 10 fold cross-validation
- Check the utility of Bagging for the present model
- Compute the scores : accuracy, f1-score
- Plot local and global explainability features with SHAP


- After these analysis for each model of classifier we will select the best one for the most relevant metric to our use case and will fine-tune it to get the best performance out of it

## SVM

## Decision Trees

In [ ]:
## Dummy classifier

from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=10)
dt.fit(tr_data, y_tr)

In [ ]:
## Hyperparameter optimization

from sklearn.model_selection import cross_val_score

cross_data = cross_val_score(DecisionTreeClassifier(), tr_data, y_tr, cv = 10)

## Random Forests

## Gradient Boosting

### Dummy classifier

### Hyperparameter optimization

### Bagging

### Explainability

### Scores

### Plots

## ANN